# Gold analytics tables

This notebook creates the four analytics-ready Gold Delta tables from the current rows of Silver. The resulting tables are the Lakehouse reporting layer. The Fabric Warehouse exposes final SQL views over these Gold tables; a second copy of Gold data is not required.

In [ ]:
from pyspark.sql import functions as F

RESOURCE_TYPES = ["Patient", "Encounter", "Observation", "Condition"]
spark.conf.set("spark.sql.session.timeZone", "UTC")

def get_json(path):
    return F.get_json_object(F.col("resource_json"), path)

def write_gold(df, table_name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(table_name))


In [ ]:
for resource_type in RESOURCE_TYPES:
    silver_table = f"silver.{resource_type}"
    gold_table = f"gold.{resource_type}"

    if not spark.catalog.tableExists(silver_table):
        raise RuntimeError(f"Missing {silver_table}; run the Silver notebook first.")

    current = spark.table(silver_table).filter(F.col("is_current") == True)

    if resource_type == "Patient":
        result = current.select(
            F.col("resource_id").alias("patient_id"),
            get_json("$.active").cast("boolean").alias("active"),
            get_json("$.gender").alias("gender"),
            F.to_date(get_json("$.birthDate")).alias("birth_date"),
            get_json("$.name[0].family").alias("family_name"),
            get_json("$.name[0].given[0]").alias("given_name"),
            get_json("$.managingOrganization.reference").alias("managing_organization_reference"),
            get_json("$.deceasedBoolean").cast("boolean").alias("deceased_boolean"),
            "source_last_updated", "extraction_timestamp", "api_url_or_params", "content_hash", "valid_from", "valid_to",
            "is_current", "ingestion_run_id", "saved_timestamp"
        )

    elif resource_type == "Encounter":
        subject_reference = get_json("$.subject.reference")
        result = current.select(
            F.col("resource_id").alias("encounter_id"),
            get_json("$.status").alias("status"),
            get_json("$.class.code").alias("class_code"),
            get_json("$.class.display").alias("class_display"),
            get_json("$.type[0].coding[0].code").alias("encounter_type_code"),
            get_json("$.type[0].coding[0].display").alias("encounter_type_display"),
            subject_reference.alias("subject_reference"),
            F.regexp_extract(subject_reference, r"Patient/(.+)$", 1).alias("patient_id"),
            get_json("$.period.start").alias("period_start"),
            get_json("$.period.end").alias("period_end"),
            get_json("$.serviceProvider.reference").alias("service_provider_reference"),
            "source_last_updated", "extraction_timestamp", "api_url_or_params", "content_hash", "valid_from", "valid_to",
            "is_current", "ingestion_run_id", "saved_timestamp"
        )

    elif resource_type == "Observation":
        subject_reference = get_json("$.subject.reference")
        effective_datetime = F.coalesce(
            get_json("$.effectiveDateTime"),
            get_json("$.effectivePeriod.start"),
            get_json("$.issued")
        )
        result = current.select(
            F.col("resource_id").alias("observation_id"),
            get_json("$.status").alias("status"),
            get_json("$.code.coding[0].code").alias("code"),
            get_json("$.code.coding[0].display").alias("code_display"),
            get_json("$.category[0].coding[0].display").alias("category"),
            subject_reference.alias("subject_reference"),
            F.regexp_extract(subject_reference, r"Patient/(.+)$", 1).alias("patient_id"),
            get_json("$.encounter.reference").alias("encounter_reference"),
            effective_datetime.alias("effective_datetime"),
            get_json("$.valueQuantity.value").cast("double").alias("value_quantity"),
            get_json("$.valueQuantity.unit").alias("value_unit"),
            get_json("$.valueString").alias("value_string"),
            get_json("$.interpretation[0].coding[0].display").alias("interpretation"),
            "source_last_updated", "extraction_timestamp", "api_url_or_params", "content_hash", "valid_from", "valid_to",
            "is_current", "ingestion_run_id", "saved_timestamp"
        )

    else:  # Condition
        subject_reference = get_json("$.subject.reference")
        result = current.select(
            F.col("resource_id").alias("condition_id"),
            get_json("$.clinicalStatus.coding[0].code").alias("clinical_status"),
            get_json("$.verificationStatus.coding[0].code").alias("verification_status"),
            get_json("$.category[0].coding[0].display").alias("category"),
            get_json("$.code.coding[0].code").alias("code"),
            get_json("$.code.coding[0].display").alias("code_display"),
            subject_reference.alias("subject_reference"),
            F.regexp_extract(subject_reference, r"Patient/(.+)$", 1).alias("patient_id"),
            get_json("$.encounter.reference").alias("encounter_reference"),
            F.coalesce(get_json("$.onsetDateTime"), get_json("$.recordedDate")).alias("onset_datetime"),
            get_json("$.recordedDate").alias("recorded_date"),
            "source_last_updated", "extraction_timestamp", "api_url_or_params", "content_hash", "valid_from", "valid_to",
            "is_current", "ingestion_run_id", "saved_timestamp"
        )

    write_gold(result, gold_table)
    print(f"{resource_type} -> {gold_table}: {spark.table(gold_table).count()} rows")
